# Early Exit Adapter Pipeline

Clean notebook wrapper around the `early_exit_adapters` package.

## 1. Install/import dependencies

In [ ]:
# If needed:
# %pip install -U torch transformers datasets pandas matplotlib tqdm wandb python-dotenv huggingface_hub accelerate safetensors

import pandas as pd

from early_exit_adapters.data import load_fineweb_stream
from early_exit_adapters.evaluate import layers_kl_over_data_rows
from early_exit_adapters.hf_utils import load_adapters_from_hf, upload_adapter_folder_to_hf
from early_exit_adapters.model import load_lm_model_and_tokenizer
from early_exit_adapters.pipeline import main
from early_exit_adapters.visualization import (
    plot_baseline_all_metrics,
    plot_baseline_metric_by_layer,
    plot_baseline_normalized_metrics,
    plot_baseline_vs_adapted,
)

## 2. Load `.env`

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 3. Load model/tokenizer

In [ ]:
model_name = "Qwen/Qwen3.5-2B"

model, tokenizer, device = load_lm_model_and_tokenizer(model_name=model_name)
device

## 4. Load FineWeb-Edu stream

In [ ]:
dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=42,
    buffer_size=10_000,
)

## 5. Train adapters

In [ ]:
adapters, logs_df = main(
    model_name="Qwen/Qwen3.5-2B",
    dataset_name="HuggingFaceFW/fineweb-edu",
    candidate_layers=[4, 8, 12, 16, 18, 20],
    max_steps=500,
    seq_len=128,
    lr=1e-4,
    temperature=2.0,
    top_k=5,
    log_every=20,
    eval_step=100,
    eval_size=32,
    save_every=100,
    use_wandb=True,
    wandb_project="qwen35-early-exit-adapters",
    wandb_run_name="qwen35_2b_residual_adapters",
    upload_to_hf=False,
    hf_repo_id="Maorb23/qwen35-2b-early-exit-adapters",
)

## 6. View `logs_df.tail()`

In [ ]:
logs_df.tail()

## 7. Run baseline/adapted layer evaluation

In [ ]:
baseline_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=42,
    buffer_size=10_000,
)

baseline_records_df, baseline_summary = layers_kl_over_data_rows(
    model=model,
    tokenizer=tokenizer,
    dataset=baseline_dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20, 22, 24],
    seq_len=128,
    num_eval_rows=100,
    temperature=2.0,
    top_k=5,
    out_dir="data/layer_kl_baseline",
)

adapted_dataset = load_fineweb_stream(
    dataset_name="HuggingFaceFW/fineweb-edu",
    split="train",
    seed=999,
    buffer_size=10_000,
)

adapted_records_df, adapted_summary = layers_kl_over_data_rows(
    model=model,
    tokenizer=tokenizer,
    dataset=adapted_dataset,
    candidate_layers=[4, 8, 12, 16, 18, 20],
    adapters=adapters,
    run_name="adapted",
    seq_len=128,
    num_eval_rows=100,
    temperature=2.0,
    top_k=5,
    out_dir="data/layer_kl_compare",
)

compare_summary = pd.concat([baseline_summary, adapted_summary], ignore_index=True)

## 8. Plot baseline metrics

In [ ]:
plot_baseline_metric_by_layer(
    baseline_summary,
    metric="kl_to_teacher",
    title="Baseline KL to final model by layer",
)
plot_baseline_all_metrics(baseline_summary)
plot_baseline_normalized_metrics(baseline_summary)

## 9. Plot baseline vs adapted metrics

In [ ]:
plot_baseline_vs_adapted(compare_summary, metric="kl_to_teacher")
plot_baseline_vs_adapted(compare_summary, metric="ce")
plot_baseline_vs_adapted(compare_summary, metric="top1_teacher_agreement")
plot_baseline_vs_adapted(compare_summary, metric="topk_overlap")
plot_baseline_vs_adapted(compare_summary, metric="accept_proxy_exact")
plot_baseline_vs_adapted(compare_summary, metric="accept_proxy_sampled")

## 10. Upload adapters to Hugging Face

In [ ]:
# upload_adapter_folder_to_hf(
#     repo_id="Maorb23/qwen35-2b-early-exit-adapters",
#     folder_path="data/early_exit_adapters",
# )

## 11. Reload adapters from Hugging Face for sanity check

In [ ]:
# reloaded_adapters = load_adapters_from_hf(
#     model=model,
#     repo_id="Maorb23/qwen35-2b-early-exit-adapters",
#     filename="adapters_final.pt",
#     device=device,
# )
# reloaded_adapters